In [4]:
import openai
import pandas as pd
import json
from openai import OpenAI
import os
import re
import time

api_key ="#########"
# Your actual OpenAI API key goes here

In [2]:
# ----------Step 1: Read the CSV/EXcel file, convert it to JSON file

#input_csv_path = "ligand_properties_extraction.csv"        # Path to your CSV file
input_xlsx_path = "Fine_tuning_dataset_v1.xlsx"   
output_jsonl_path = "ligand_finetune_dataset.jsonl"         # Output .jsonl file

# === LOAD CSV ===
#df = pd.read_csv(input_csv_path)
df = pd.read_excel(input_xlsx_path)
df = df.iloc[:, :-2]

# === CONVERT AND SAVE ===
with open(output_jsonl_path, "w", encoding="utf-8") as fout:
    current_paragraph = None
    group = []

    for _, row in df.iterrows():
        paragraph = row["Paragraph"]
        
        # Detect start of new paragraph
        if pd.notna(paragraph) and paragraph.strip():
            if group:
                # Write last paragraph's message pair
                fout.write(json.dumps({
                    "messages": [
                        {"role": "user", "content": current_paragraph},
                        {"role": "assistant", "content": json.dumps(group, indent=2)}
                    ]
                }) + "\n")
                group = []
            current_paragraph = paragraph

        # Check if there’s meaningful ligand data
        if any(pd.notna(row[col]) and str(row[col]).strip() for col in df.columns[1:]):
            ligand_data = {
                col.lower().replace(" ", "_"): row[col] if pd.notna(row[col]) else ""
                for col in df.columns[1:]
            }
            group.append(ligand_data)

    # Write final message pair
    if group:
        fout.write(json.dumps({
            "messages": [
                {"role": "user", "content": current_paragraph},
                {"role": "assistant", "content": json.dumps(group, indent=2)}
            ]
        }) + "\n")

print(f"✅ Converted to {output_jsonl_path}")


✅ Converted to ligand_finetune_dataset.jsonl


In [4]:
# ----------Step 2: Optional, reading the JSONL file to python pandas data frame

# Path to your .jsonl file
jsonl_path = "ligand_finetune_dataset.jsonl"

# Load and flatten the file
flattened_data = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        paragraph = entry["messages"][0]["content"]
        try:
            ligand_list = json.loads(entry["messages"][1]["content"])
            for ligand in ligand_list:
                flattened_row = {
                    "Paragraph": paragraph,
                    **ligand  # Unpack ligand properties into columns
                }
                flattened_data.append(flattened_row)
        except json.JSONDecodeError:
            print("⚠️ Warning: Failed to parse assistant output as JSON")

# Convert to DataFrame
df = pd.DataFrame(flattened_data)

# Display or save
print(df.head())
# df.to_csv("flattened_ligand_properties.csv", index=False)


                                           Paragraph    ligand_name  \
0  One of the first examples of DGA based extract...        DMDODGA   
1  One of the first examples of DGA based extract...          TODGA   
2  One of the first examples of DGA based extract...  DGA (general)   
3  The extraction of 75 elements by N,N,N ,N -tet...          TODGA   
4  For the reprocessing of spent nuclear fuel and...            TBP   

                                 chemical_properties radiolysis_resistance  \
0                                                                            
1  Contains amidic octyl chains; due to size, com...                         
2  Contains variable lengths of amidic alkyl chai...                         
3  Extracted chemical forms determined as M(TODGA...                         
4                                                                            

                                      thermodynamics kinetics  \
0                                      

In [ ]:
# ----------Step 3: uploaded the JSON file for Fine-tuning the GPT model
# export OPENAI_API_KEY="your-api-key"
client = OpenAI(api_key= api_key)  # or replace with string if preferred

# ---------- Step 3.1: Upload JSONL training file ----------
file_path = "ligand_finetune_dataset.jsonl"
with open(file_path, "rb") as f:
    upload = client.files.create(file=f, purpose="fine-tune")

print("Uploaded file ID:", upload.id)

# ---------- Step 3.2: Create fine-tuning job ----------
# uncomment the below line if you want to finetune the GPT model.
# fine_tune_job = client.fine_tuning.jobs.create(
    training_file=upload.id,
    model="gpt-3.5-turbo"  # or another supported base model
)

print("Fine-tuning job ID:", fine_tune_job.id)

# ---------- Step 3.3: Monitor fine-tuning job ----------
job_id = fine_tune_job.id
job_status = client.fine_tuning.jobs.retrieve(job_id)
print("Job status:", job_status.status)

# ---------- Step 3.4(Optional): List all fine-tuned models ----------
models = client.fine_tuning.jobs.list(limit=10)
for job in models.data:
    print(job.fine_tuned_model)



In [3]:
# ---------- Step 3.5: Optional,  Check the runing status(model is finished or not), and get the model ID----------¶
# if you see "Status: running, Model: None", which means the fine-tune is not finished yet, 
# When you finished fine-tuning, you will see Status:succeeded, Monel: XXXX. you also get an email telling you the fine-tune is finished.

client = OpenAI(api_key=api_key)

models = client.fine_tuning.jobs.list()

for job in models.data:
    print(f"Job ID: {job.id}, Status: {job.status}, Model: {job.fine_tuned_model}")

Job ID: ftjob-f0nN873hW6FWynarzmC9e7E9, Status: succeeded, Model: ft:gpt-3.5-turbo-0125:uc-berkeley::CXF5BaWv
Job ID: ftjob-KHwoDUBySPFS5DcGJGJCZCpr, Status: succeeded, Model: ft:gpt-3.5-turbo-0125:uc-berkeley::Bq5fHucH
Job ID: ftjob-g9s1KZH8amTVE0Nm31aUTSq2, Status: succeeded, Model: ft:gpt-3.5-turbo-0125:uc-berkeley::Bq5U06mX
Job ID: ftjob-G01ADka5VIPm3qfT6qbK3ljC, Status: succeeded, Model: ft:gpt-3.5-turbo-0125:uc-berkeley::Bq5HUS7K
Job ID: ftjob-CcqeL3MSSAq0nGuTGwwl6Jyg, Status: succeeded, Model: ft:gpt-3.5-turbo-0125:uc-berkeley::BpVZrjBD


#### To save volumne, WE don't need input paragraphs in the database. But we need the input paragraphs for the extraction result grading.
## In Step 4 (below cell):
### to create database, keep the line "df = df.iloc[:, 1:] " to remove the input paragraphs, 
### to grade the extraction result, comment that line, to keep the input paragphs

file_path = r".\Test_set\paper1.txt"#"Paragraphs_with_multiple_ligands.txt"          # input paragraphs
fine_tuned_model_id = "ft:gpt-3.5-turbo-0125:uc-berkeley::CXF5BaWv"# for 737 paragraphs

#fine_tuned_model_id = "ft:gpt-3.5-turbo-0125:uc-berkeley::Bq5fHucH"# for 100 paragraphs, it seems that this model is better. 


output_excel = r".\Test_set\paper1.xlsx"
output_jsonl = r".\Test_set\paper1.jsonl"
temperature = 0
# =================================================================
# ---- Load paragraphs ---------------------------------------------
def load_paragraphs(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    # each non-empty line is treated as one paragraph
    return [p.strip() for p in text.split("\n") if p.strip()]
paragraphs = load_paragraphs(file_path)

all_results = []
len(paragraphs)

## Step 4: Test the fine-tuned model

In [4]:
#---------- Step 4: Test the fine-tuned model ----------¶
# Your fine-tuned model ID and file path
# ==== CONFIG ======================================================
file_path = r".\Test_set\paper3.txt"
fine_tuned_model_id = "ft:gpt-3.5-turbo-0125:uc-berkeley::CXF5BaWv"
output_excel = r".\Test_set\paper3.xlsx"
output_jsonl = r".\Test_set\paper3.jsonl"
temperature = 0
# =================================================================

client = OpenAI(api_key=api_key)

def load_paragraphs(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    return [p.strip() for p in text.split("\n") if p.strip()]

def clean_result_text(text):
    """Clean common model formatting issues before JSON parsing."""
    text = text.strip()

    # Remove markdown fences
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
        if text.endswith("```"):
            text = text[:-3].strip()
    elif text.startswith("```"):
        text = text[3:].strip()
        if text.endswith("```"):
            text = text[:-3].strip()

    # Fix common invalid escape sequence seen in model output
    text = text.replace(r"\N{SUPERSCRIPT TWO}", "²")

    return text

def extract_first_json_object(text):
    """
    Extract the first balanced JSON object from text.
    Useful if the model adds extra garbage before/after the JSON.
    """
    start = text.find("{")
    if start == -1:
        return text

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i+1]

    return text

def parse_model_json(result_text):
    """
    Parse model output robustly.
    Expected format:
    {
      "items": [...]
    }
    """
    result_text = clean_result_text(result_text)

    # First try direct parse
    try:
        parsed = json.loads(result_text)
        if isinstance(parsed, dict) and "items" in parsed:
            return parsed["items"]
        raise ValueError("Parsed JSON is not an object with key 'items'.")
    except Exception:
        pass

    # Fallback: extract first JSON object
    candidate = extract_first_json_object(result_text)
    candidate = clean_result_text(candidate)
    parsed = json.loads(candidate)

    if isinstance(parsed, dict) and "items" in parsed:
        return parsed["items"]

    raise ValueError("Could not parse JSON object with key 'items'.")

paragraphs = load_paragraphs(file_path)
all_results = []

for idx, para in enumerate(paragraphs, 1):
    print(f"Processing paragraph {idx}/{len(paragraphs)}")

    system_prompt = (
        "You extract and classify ligand properties into a structured format. "
        "Return exactly one JSON object with one key: 'items'. "
        "The value of 'items' must be an array of dictionaries. "
        "Do not output markdown, explanation, or any extra text."
    )

    user_prompt = (
        "Extract ligand data from the paragraph below.\n\n"
        "Return strictly this JSON object format:\n"
        "{\n"
        '  "items": [\n'
        "    {\n"
        '      "Ligand_Name": "...",\n'
        '      "Chemical_Properties": "...",\n'
        '      "Radiolysis_Resistance": "...",\n'
        '      "Thermodynamics": "...",\n'
        '      "Kinetics": "...",\n'
        '      "Loading_Capacity": "...",\n'
        '      "Operational_Condition_Range": "...",\n'
        '      "Physical_Properties": "...",\n'
        '      "Phase_Disengagement": "...",\n'
        '      "Extraction_Efficiency": "...",\n'
        '      "Separation_Factor": "...",\n'
        '      "Dependence": "..."\n'
        "    }\n"
        "  ]\n"
        "}\n\n"
        "Rules:\n"
        "- Return exactly one JSON object.\n"
        "- If nothing is mentioned, use empty strings.\n"
        "- If multiple ligands exist, return multiple dictionaries inside items.\n"
        "- Do not use invalid escapes like \\N{...}.\n"
        "- Use plain Unicode characters directly, such as ² instead of escape tricks.\n\n"
        "Paragraph:\n"
        f"{para}"
    )

    try:
        response = client.chat.completions.create(
            model=fine_tuned_model_id,
            temperature=temperature,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        result_text = response.choices[0].message.content.strip()
        parsed_items = parse_model_json(result_text)

        """if isinstance(parsed_items, list):
            for rec in parsed_items:
                row = {
                    "Paragraph": para,
                    "Ligand_Name": rec.get("ligand_name", ""),
                    "Chemical_Properties": rec.get("chemical_properties", ""),
                    "Radiolysis_Resistance": rec.get("radiolysis_resistance", ""),
                    "Thermodynamics": rec.get("thermodynamics", ""),
                    "Kinetics": rec.get("kinetics", ""),
                    "Loading_Capacity": rec.get("loading_capacity", ""),
                    "Operational_Condition_Range": rec.get("operational_condition_range", ""),
                    "Physical_Properties": rec.get("physical_properties", ""),
                    "Phase_Disengagement": rec.get("phase_disengagement", ""),
                    "Extraction_Efficiency": rec.get("extraction_efficiency", ""),
                    "Separation_Factor": rec.get("separation_factor", ""),
                    "Dependence": rec.get("dependence", ""),
                }
                all_results.append(row)"""
        if isinstance(parsed_items, list):
            if len(parsed_items) == 0:
                all_results.append({
                    "Paragraph": para,
                    "Ligand_Name": "",
                    "Chemical_Properties": "",
                    "Radiolysis_Resistance": "",
                    "Thermodynamics": "",
                    "Kinetics": "",
                    "Loading_Capacity": "",
                    "Operational_Condition_Range": "",
                    "Physical_Properties": "",
                    "Phase_Disengagement": "",
                    "Extraction_Efficiency": "",
                    "Separation_Factor": "",
                    "Dependence": "",
                })
            else:
                for rec in parsed_items:
                    row = {
                        "Paragraph": para,
                        "Ligand_Name": rec.get("ligand_name", ""),
                        "Chemical_Properties": rec.get("chemical_properties", ""),
                        "Radiolysis_Resistance": rec.get("radiolysis_resistance", ""),
                        "Thermodynamics": rec.get("thermodynamics", ""),
                        "Kinetics": rec.get("kinetics", ""),
                        "Loading_Capacity": rec.get("loading_capacity", ""),
                        "Operational_Condition_Range": rec.get("operational_condition_range", ""),
                        "Physical_Properties": rec.get("physical_properties", ""),
                        "Phase_Disengagement": rec.get("phase_disengagement", ""),
                        "Extraction_Efficiency": rec.get("extraction_efficiency", ""),
                        "Separation_Factor": rec.get("separation_factor", ""),
                        "Dependence": rec.get("dependence", ""),
                    }
                    all_results.append(row)

        else:
            print(f"Expected list in parsed_items, got {type(parsed_items)}")

    except Exception as e:
        print(f"Failed on paragraph {idx}: {e}")
        print("Raw response:\n", result_text if "result_text" in locals() else "No response text")
        continue

df = pd.DataFrame(all_results)

columns_order = [
    "Paragraph",
    "Ligand_Name",
    "Chemical_Properties",
    "Radiolysis_Resistance",
    "Thermodynamics",
    "Kinetics",
    "Loading_Capacity",
    "Operational_Condition_Range",
    "Physical_Properties",
    "Phase_Disengagement",
    "Extraction_Efficiency",
    "Separation_Factor",
    "Dependence",
]
df = df.reindex(columns=columns_order)

rows = []
last_para = None
for _, r in df.iterrows():
    para = r["Paragraph"]
    if para != last_para:
        last_para = para
        rows.append(r)
    else:
        r2 = r.copy()
        r2["Paragraph"] = ""
        rows.append(r2)

df = pd.DataFrame(rows, columns=columns_order)

df.to_excel(output_excel, index=False)
print(f"Saved Excel: {output_excel}")

with open(output_jsonl, "w", encoding="utf-8") as f:
    for record in df.to_dict(orient="records"):
        json.dump(record, f, ensure_ascii=False)
        f.write("\n")
print(f"Saved JSONL: {output_jsonl}")

Processing paragraph 1/26
Processing paragraph 2/26
Processing paragraph 3/26
Processing paragraph 4/26
Processing paragraph 5/26
Processing paragraph 6/26
Processing paragraph 7/26
Processing paragraph 8/26
Processing paragraph 9/26
Processing paragraph 10/26
Processing paragraph 11/26
Processing paragraph 12/26
Processing paragraph 13/26
Processing paragraph 14/26
Processing paragraph 15/26
Processing paragraph 16/26
Processing paragraph 17/26
Processing paragraph 18/26
Processing paragraph 19/26
Processing paragraph 20/26
Processing paragraph 21/26
Processing paragraph 22/26
Processing paragraph 23/26
Processing paragraph 24/26
Processing paragraph 25/26
Processing paragraph 26/26
Saved Excel: .\Test_set\paper3.xlsx
Saved JSONL: .\Test_set\paper3.jsonl


## Grading Section

In [7]:
# updated on March 31, compare ligand in a more smart way, rather than row by row.
import time
import re
import difflib
import pandas as pd
import openai

# ---------- CONFIG ----------
MODEL = "gpt-4o-mini"
SLEEP = 0.3
INPUT_GT = r".\Test_set\paper3_temp0_EAE_proofed.xlsx"# Answer, ground truth.
INPUT_PRED = r".\Test_set\paper3_extracted_Prompt_method.xlsx"# predicted by model
OUTPUT_FILE = r".\Test_set\paper3_Prompt_grades_gpt.xlsx"
CHECKPOINT_EVERY = 20   # checkpoint by paragraph count

# ---------- GPT PROMPTS ----------
SYSTEM_PROMPT = """You are an expert grader in solvent extraction chemistry and nuclear separations.
Grade a SINGLE property (0..1) for the given ligand in the current paragraph.

Rules:
- Grade based on MEANING (scientific equivalence), not wording.
- Treat synonyms as equivalent (e.g., "stable under irradiation" = "minimal degradation").
- Normalize units and numbers (Gy↔kGy, mol/L≈M, °C ignored).
- If both imply the same meaning and numeric range (within ±10%), score 1.0.
- If both are completely blank or missing, score 1.0.

Scoring rules (for numeric or semantic similarity):
- ±10–20% → 0.9–0.95
- ±20–40% → 0.7–0.8
- ±40–70% → 0.4–0.6
- >70% → 0.0–0.3
- Directionally opposite → 0.0
- Same qualitative meaning → 0.9–1.0
- Weak overlap but not contradictory → 0.4–0.7
- Clearly wrong or unrelated → 0.0

Output ONLY the numeric value (no text).
"""

def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() == "nan":
        return ""
    return s

def norm_name(s):
    s = clean_text(s).lower()
    s = re.sub(r"\s+", " ", s)
    return s

def grade_property(client, paragraph, ligand, prop, gt, pred):
    gt = clean_text(gt)
    pred = clean_text(pred)

    if gt == "" and pred == "":
        return 1.0

    user_prompt = f"""Paragraph:
{paragraph}

Ligand:
{ligand}

Property:
{prop}

Ground Truth Value:
{gt}

Predicted Value:
{pred}

Return ONLY a numeric score between 0 and 1."""
    
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=10,
    )

    txt = clean_text(resp.choices[0].message.content)
    try:
        return float(txt)
    except Exception:
        return None

def prepare_dataframe(df):
    df = df.copy().reset_index(drop=True)

    if "Paragraph" not in df.columns:
        raise ValueError("Missing required column: Paragraph")
    if "Ligand_Name" not in df.columns:
        raise ValueError("Missing required column: Ligand_Name")

    for col in df.columns:
        df[col] = df[col].apply(clean_text)

    df["Paragraph_start"] = df["Paragraph"].apply(lambda x: clean_text(x) != "")
    return df

def split_into_ordered_paragraph_groups(df):
    """
    Split dataframe into paragraph groups by row order.
    A new group starts whenever Paragraph cell is non-empty.
    """
    start_indices = df.index[df["Paragraph_start"]].tolist()

    if not start_indices:
        raise ValueError("No paragraph starts found. The 'Paragraph' column may be empty.")

    groups = []
    for k, start_idx in enumerate(start_indices):
        end_idx = start_indices[k + 1] if k + 1 < len(start_indices) else len(df)
        group = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        paragraph_text = clean_text(group.loc[0, "Paragraph"])
        group["Paragraph_full"] = paragraph_text
        groups.append(group)

    return groups

def match_ligands(gt_group, pred_group):
    """
    Return:
      matched_pairs: list of (gt_idx, pred_idx)
      unmatched_gt: list of gt_idx
      unmatched_pred: list of pred_idx
    """
    matched_pairs = []
    unmatched_gt = list(gt_group.index)
    unmatched_pred = list(pred_group.index)

    gt_names = {i: norm_name(gt_group.at[i, "Ligand_Name"]) for i in gt_group.index}
    pred_names = {j: norm_name(pred_group.at[j, "Ligand_Name"]) for j in pred_group.index}

    # Pass 1: exact name match
    used_pred = set()
    for i in list(unmatched_gt):
        gname = gt_names[i]
        candidates = [j for j in unmatched_pred if pred_names[j] == gname and j not in used_pred]
        if candidates:
            j = candidates[0]
            matched_pairs.append((i, j))
            used_pred.add(j)

    matched_gt = {i for i, _ in matched_pairs}
    matched_pred = {j for _, j in matched_pairs}
    unmatched_gt = [i for i in unmatched_gt if i not in matched_gt]
    unmatched_pred = [j for j in unmatched_pred if j not in matched_pred]

    # Pass 2: fuzzy name match
    additional_matches = []
    for i in unmatched_gt:
        gname = gt_names[i]
        best_j = None
        best_score = 0.0

        for j in unmatched_pred:
            pname = pred_names[j]
            score = difflib.SequenceMatcher(None, gname, pname).ratio()
            if score > best_score:
                best_score = score
                best_j = j

        if best_j is not None and best_score >= 0.88:
            additional_matches.append((i, best_j))

    used_pred = set()
    filtered = []
    for i, j in additional_matches:
        if j not in used_pred:
            filtered.append((i, j))
            used_pred.add(j)

    matched_pairs.extend(filtered)

    matched_gt = {i for i, _ in matched_pairs}
    matched_pred = {j for _, j in matched_pairs}
    unmatched_gt = [i for i in gt_group.index if i not in matched_gt]
    unmatched_pred = [j for j in pred_group.index if j not in matched_pred]

    return matched_pairs, unmatched_gt, unmatched_pred

# ---------- MAIN ----------
client = openai.OpenAI(api_key=api_key)

df_gt = pd.read_excel(INPUT_GT)
df_pred = pd.read_excel(INPUT_PRED)

df_gt = prepare_dataframe(df_gt)
df_pred = prepare_dataframe(df_pred)

print("GT rows:", len(df_gt))
print("PRED rows:", len(df_pred))

property_cols = [c for c in df_gt.columns if c not in ["Paragraph", "Paragraph_start", "Paragraph_full", "Ligand_Name"]]

gt_groups = split_into_ordered_paragraph_groups(df_gt)
pred_groups = split_into_ordered_paragraph_groups(df_pred)

print("GT paragraphs:", len(gt_groups))
print("PRED paragraphs:", len(pred_groups))

if len(gt_groups) != len(pred_groups):
    raise ValueError(
        f"Paragraph count mismatch: GT has {len(gt_groups)} paragraphs, "
        f"prediction has {len(pred_groups)} paragraphs"
    )

graded_rows = []
extra_pred_rows = []

for p_idx, (gt_group, pred_group) in enumerate(zip(gt_groups, pred_groups), start=1):
    paragraph = clean_text(gt_group.loc[0, "Paragraph_full"])

    matched_pairs, unmatched_gt, unmatched_pred = match_ligands(gt_group, pred_group)

    print(f"\n[Paragraph {p_idx}]")
    print("GT ligands:", gt_group["Ligand_Name"].tolist())
    print("Pred ligands:", pred_group["Ligand_Name"].tolist())

    # matched rows
    for gt_i, pred_i in matched_pairs:
        gt_row = gt_group.loc[gt_i]
        pred_row = pred_group.loc[pred_i]

        out = {
            "Paragraph": paragraph,
            "GT Ligand_Name": clean_text(gt_row.get("Ligand_Name", "")),
            "Pred Ligand_Name": clean_text(pred_row.get("Ligand_Name", "")),
            "Match Status": "matched"
        }

        row_scores = []
        for prop in property_cols:
            gt_val = clean_text(gt_row.get(prop, ""))
            pred_val = clean_text(pred_row.get(prop, ""))

            score = grade_property(
                client=client,
                paragraph=paragraph,
                ligand=clean_text(gt_row.get("Ligand_Name", "")),
                prop=prop,
                gt=gt_val,
                pred=pred_val
            )

            out[prop] = score
            row_scores.append(score if score is not None else 0.0)

            print(f"[Paragraph {p_idx}] Matched | {out['GT Ligand_Name']} | {prop} -> {score}")
            time.sleep(SLEEP)

        out["Row Average"] = sum(row_scores) / len(row_scores) if row_scores else None
        graded_rows.append(out)

    # missing ligands in prediction
    for gt_i in unmatched_gt:
        gt_row = gt_group.loc[gt_i]

        out = {
            "Paragraph": paragraph,
            "GT Ligand_Name": clean_text(gt_row.get("Ligand_Name", "")),
            "Pred Ligand_Name": "",
            "Match Status": "missing_in_prediction"
        }

        for prop in property_cols:
            out[prop] = 0.0

        out["Row Average"] = 0.0
        graded_rows.append(out)

        print(f"[Paragraph {p_idx}] Missing ligand in prediction -> {out['GT Ligand_Name']}")

    # extra ligands in prediction
    for pred_i in unmatched_pred:
        pred_row = pred_group.loc[pred_i]

        extra = {
            "Paragraph": paragraph,
            "GT Ligand_Name": "",
            "Pred Ligand_Name": clean_text(pred_row.get("Ligand_Name", "")),
            "Match Status": "extra_in_prediction"
        }

        for prop in property_cols:
            extra[prop] = ""

        extra["Row Average"] = ""
        extra_pred_rows.append(extra)

        print(f"[Paragraph {p_idx}] Extra ligand in prediction -> {extra['Pred Ligand_Name']}")

    # checkpoint
    if p_idx % CHECKPOINT_EVERY == 0:
        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
            pd.DataFrame(graded_rows).to_excel(writer, sheet_name="graded", index=False)
            pd.DataFrame(extra_pred_rows).to_excel(writer, sheet_name="extra_pred", index=False)
        print(f"[Checkpoint] Saved progress up to paragraph {p_idx} -> {OUTPUT_FILE}")

# ---------- final save ----------
graded_df = pd.DataFrame(graded_rows)
extra_df = pd.DataFrame(extra_pred_rows)

summary = {
    "Total GT paragraphs": len(gt_groups),
    "Total GT rows": len(df_gt),
    "Total Pred rows": len(df_pred),
    "Matched rows": int((graded_df["Match Status"] == "matched").sum()) if not graded_df.empty else 0,
    "Missing GT ligands": int((graded_df["Match Status"] == "missing_in_prediction").sum()) if not graded_df.empty else 0,
    "Extra predicted ligands": len(extra_df),
}

if not graded_df.empty and "Row Average" in graded_df.columns:
    valid_scores = pd.to_numeric(graded_df["Row Average"], errors="coerce")
    summary["Overall average score"] = valid_scores.mean()

summary_df = pd.DataFrame([summary])

# ---------- ADD AVERAGE ROWS ----------

if not graded_df.empty:
    avg_row = {
        "Paragraph": "AVERAGE_PER_COLUMN",
        "GT Ligand_Name": "",
        "Pred Ligand_Name": "",
        "Match Status": ""
    }

    # compute column-wise averages
    for prop in property_cols:
        col_vals = pd.to_numeric(graded_df[prop], errors="coerce")
        avg_row[prop] = col_vals.mean()

    # row average column
    if "Row Average" in graded_df.columns:
        avg_row["Row Average"] = pd.to_numeric(
            graded_df["Row Average"], errors="coerce"
        ).mean()

    # overall average across ALL grade cells
    all_scores = []
    for prop in property_cols:
        vals = pd.to_numeric(graded_df[prop], errors="coerce").dropna()
        all_scores.extend(vals.tolist())

    overall_avg = sum(all_scores) / len(all_scores) if all_scores else None

    overall_row = {
        "Paragraph": "OVERALL_AVERAGE_ALL_CELLS",
        "GT Ligand_Name": "",
        "Pred Ligand_Name": "",
        "Match Status": ""
    }

    for prop in property_cols:
        overall_row[prop] = ""

    overall_row["Row Average"] = overall_avg

    # append to dataframe
    graded_df = pd.concat(
        [graded_df, pd.DataFrame([avg_row, overall_row])],
        ignore_index=True
    )

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    graded_df.to_excel(writer, sheet_name="graded", index=False)
    extra_df.to_excel(writer, sheet_name="extra_pred", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)

print(f"\n✅ Finished. Saved graded Excel to: {OUTPUT_FILE}")

GT rows: 32
PRED rows: 32
GT paragraphs: 26
PRED paragraphs: 26

[Paragraph 1]
GT ligands: ['octadecyl acyclopa (ODA)']
Pred ligands: ['octadecyl acyclopa (ODA)']
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Chemical_Properties -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Radiolysis_Resistance -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Thermodynamics -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Kinetics -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Loading_Capacity -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Operational_Condition_Range -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Physical_Properties -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Phase_Disengagement -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Extraction_Efficiency -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Separation_Factor -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Dependence -> 0

In [8]:
import time
import re
import difflib
import pandas as pd
import openai

# ---------- CONFIG ----------
MODEL = "gpt-4o-mini"
SLEEP = 0.3
INPUT_GT = r".\Test_set\paper3_temp0_EAE_proofed.xlsx"# Answer, ground truth.
INPUT_PRED = r".\Test_set\paper3_extracted_Prompt_method.xlsx"# predicted by model
OUTPUT_FILE = r".\Test_set\paper3_Prompt_grades_gpt.xlsx"
CHECKPOINT_EVERY = 20   # checkpoint by paragraph count

# ---------- GPT PROMPTS ----------
SYSTEM_PROMPT = """You are an expert grader in solvent extraction chemistry and nuclear separations.
Grade a SINGLE property (0..1) for the given ligand in the current paragraph.

Rules:
- Grade based on MEANING (scientific equivalence), not wording.
- Treat synonyms as equivalent (e.g., "stable under irradiation" = "minimal degradation").
- Normalize units and numbers (Gy↔kGy, mol/L≈M, °C ignored).
- If both imply the same meaning and numeric range (within ±10%), score 1.0.
- If both are completely blank or missing, score 1.0.

Scoring rules (for numeric or semantic similarity):
- ±10–20% → 0.9–0.95
- ±20–40% → 0.7–0.8
- ±40–70% → 0.4–0.6
- >70% → 0.0–0.3
- Directionally opposite → 0.0
- Same qualitative meaning → 0.9–1.0
- Weak overlap but not contradictory → 0.4–0.7
- Clearly wrong or unrelated → 0.0

Output ONLY the numeric value (no text).
"""

def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() == "nan":
        return ""
    return s

def norm_name(s):
    s = clean_text(s).lower()
    s = re.sub(r"\s+", " ", s)
    return s

def grade_property(client, paragraph, ligand, prop, gt, pred):
    gt = clean_text(gt)
    pred = clean_text(pred)

    if gt == "" and pred == "":
        return 1.0

    user_prompt = f"""Paragraph:
{paragraph}

Ligand:
{ligand}

Property:
{prop}

Ground Truth Value:
{gt}

Predicted Value:
{pred}

Return ONLY a numeric score between 0 and 1."""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=10,
    )

    txt = clean_text(resp.choices[0].message.content)
    try:
        return float(txt)
    except Exception:
        return None

def prepare_dataframe(df):
    df = df.copy().reset_index(drop=True)

    if "Paragraph" not in df.columns:
        raise ValueError("Missing required column: Paragraph")
    if "Ligand_Name" not in df.columns:
        raise ValueError("Missing required column: Ligand_Name")

    for col in df.columns:
        df[col] = df[col].apply(clean_text)

    df["Paragraph_start"] = df["Paragraph"].apply(lambda x: clean_text(x) != "")
    return df

def split_into_ordered_paragraph_groups(df):
    """
    Split dataframe into paragraph groups by row order.
    A new group starts whenever Paragraph cell is non-empty.
    """
    start_indices = df.index[df["Paragraph_start"]].tolist()

    if not start_indices:
        raise ValueError("No paragraph starts found. The 'Paragraph' column may be empty.")

    groups = []
    for k, start_idx in enumerate(start_indices):
        end_idx = start_indices[k + 1] if k + 1 < len(start_indices) else len(df)
        group = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

        paragraph_text = clean_text(group.loc[0, "Paragraph"])
        group["Paragraph_full"] = paragraph_text
        groups.append(group)

    return groups

def match_ligands(gt_group, pred_group):
    """
    Return:
      matched_pairs: list of (gt_idx, pred_idx)
      unmatched_gt: list of gt_idx
      unmatched_pred: list of pred_idx
    """
    matched_pairs = []
    unmatched_gt = list(gt_group.index)
    unmatched_pred = list(pred_group.index)

    gt_names = {i: norm_name(gt_group.at[i, "Ligand_Name"]) for i in gt_group.index}
    pred_names = {j: norm_name(pred_group.at[j, "Ligand_Name"]) for j in pred_group.index}

    # Pass 1: exact name match
    used_pred = set()
    for i in list(unmatched_gt):
        gname = gt_names[i]
        candidates = [j for j in unmatched_pred if pred_names[j] == gname and j not in used_pred]
        if candidates:
            j = candidates[0]
            matched_pairs.append((i, j))
            used_pred.add(j)

    matched_gt = {i for i, _ in matched_pairs}
    matched_pred = {j for _, j in matched_pairs}
    unmatched_gt = [i for i in unmatched_gt if i not in matched_gt]
    unmatched_pred = [j for j in unmatched_pred if j not in matched_pred]

    # Pass 2: fuzzy name match
    additional_matches = []
    for i in unmatched_gt:
        gname = gt_names[i]
        best_j = None
        best_score = 0.0

        for j in unmatched_pred:
            pname = pred_names[j]
            score = difflib.SequenceMatcher(None, gname, pname).ratio()
            if score > best_score:
                best_score = score
                best_j = j

        if best_j is not None and best_score >= 0.88:
            additional_matches.append((i, best_j))

    used_pred = set()
    filtered = []
    for i, j in additional_matches:
        if j not in used_pred:
            filtered.append((i, j))
            used_pred.add(j)

    matched_pairs.extend(filtered)

    matched_gt = {i for i, _ in matched_pairs}
    matched_pred = {j for _, j in matched_pairs}
    unmatched_gt = [i for i in gt_group.index if i not in matched_gt]
    unmatched_pred = [j for j in pred_group.index if j not in matched_pred]

    return matched_pairs, unmatched_gt, unmatched_pred

# ---------- MAIN ----------
client = openai.OpenAI(api_key=api_key)

df_gt = pd.read_excel(INPUT_GT)
df_pred = pd.read_excel(INPUT_PRED)

df_gt = prepare_dataframe(df_gt)
df_pred = prepare_dataframe(df_pred)

print("GT rows:", len(df_gt))
print("PRED rows:", len(df_pred))

property_cols = [
    c for c in df_gt.columns
    if c not in ["Paragraph", "Paragraph_start", "Paragraph_full", "Ligand_Name"]
]

grade_cols = ["Ligand_Name_Grade"] + property_cols

gt_groups = split_into_ordered_paragraph_groups(df_gt)
pred_groups = split_into_ordered_paragraph_groups(df_pred)

print("GT paragraphs:", len(gt_groups))
print("PRED paragraphs:", len(pred_groups))

if len(gt_groups) != len(pred_groups):
    raise ValueError(
        f"Paragraph count mismatch: GT has {len(gt_groups)} paragraphs, "
        f"prediction has {len(pred_groups)} paragraphs"
    )

graded_rows = []
extra_pred_rows = []

for p_idx, (gt_group, pred_group) in enumerate(zip(gt_groups, pred_groups), start=1):
    paragraph = clean_text(gt_group.loc[0, "Paragraph_full"])

    matched_pairs, unmatched_gt, unmatched_pred = match_ligands(gt_group, pred_group)

    print(f"\n[Paragraph {p_idx}]")
    print("GT ligands:", gt_group["Ligand_Name"].tolist())
    print("Pred ligands:", pred_group["Ligand_Name"].tolist())

    # matched rows
    for gt_i, pred_i in matched_pairs:
        gt_row = gt_group.loc[gt_i]
        pred_row = pred_group.loc[pred_i]

        gt_name = clean_text(gt_row.get("Ligand_Name", ""))
        pred_name = clean_text(pred_row.get("Ligand_Name", ""))

        out = {
            "Paragraph": paragraph,
            "GT Ligand_Name": gt_name,
            "Pred Ligand_Name": pred_name,
            "Match Status": "matched",
            "Ligand_Name_Grade": 1.0
        }

        row_scores = [1.0]   # include ligand-name grade

        for prop in property_cols:
            gt_val = clean_text(gt_row.get(prop, ""))
            pred_val = clean_text(pred_row.get(prop, ""))

            score = grade_property(
                client=client,
                paragraph=paragraph,
                ligand=gt_name,
                prop=prop,
                gt=gt_val,
                pred=pred_val
            )

            out[prop] = score
            row_scores.append(score if score is not None else 0.0)

            print(f"[Paragraph {p_idx}] Matched | {gt_name} | {prop} -> {score}")
            time.sleep(SLEEP)

        out["Row Average"] = sum(row_scores) / len(row_scores) if row_scores else None
        graded_rows.append(out)

    # missing ligands in prediction
    for gt_i in unmatched_gt:
        gt_row = gt_group.loc[gt_i]

        out = {
            "Paragraph": paragraph,
            "GT Ligand_Name": clean_text(gt_row.get("Ligand_Name", "")),
            "Pred Ligand_Name": "",
            "Match Status": "missing_in_prediction",
            "Ligand_Name_Grade": 0.0
        }

        for prop in property_cols:
            out[prop] = 0.0

        out["Row Average"] = 0.0
        graded_rows.append(out)

        print(f"[Paragraph {p_idx}] Missing ligand in prediction -> {out['GT Ligand_Name']}")

    # extra ligands in prediction
    for pred_i in unmatched_pred:
        pred_row = pred_group.loc[pred_i]

        extra = {
            "Paragraph": paragraph,
            "GT Ligand_Name": "",
            "Pred Ligand_Name": clean_text(pred_row.get("Ligand_Name", "")),
            "Match Status": "extra_in_prediction",
            "Ligand_Name_Grade": 0.0
        }

        for prop in property_cols:
            extra[prop] = ""

        extra["Row Average"] = ""
        extra_pred_rows.append(extra)

        print(f"[Paragraph {p_idx}] Extra ligand in prediction -> {extra['Pred Ligand_Name']}")

    # checkpoint
    if p_idx % CHECKPOINT_EVERY == 0:
        temp_graded_df = pd.DataFrame(graded_rows)
        temp_extra_df = pd.DataFrame(extra_pred_rows)

        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
            temp_graded_df.to_excel(writer, sheet_name="graded", index=False)
            temp_extra_df.to_excel(writer, sheet_name="extra_pred", index=False)
        print(f"[Checkpoint] Saved progress up to paragraph {p_idx} -> {OUTPUT_FILE}")

# ---------- BUILD FINAL DATAFRAMES ----------
graded_df = pd.DataFrame(graded_rows)
extra_df = pd.DataFrame(extra_pred_rows)

# ---------- ADD AVERAGE ROWS ----------
if not graded_df.empty:
    avg_row = {
        "Paragraph": "AVERAGE_PER_COLUMN",
        "GT Ligand_Name": "",
        "Pred Ligand_Name": "",
        "Match Status": ""
    }

    for col in grade_cols:
        col_vals = pd.to_numeric(graded_df[col], errors="coerce")
        avg_row[col] = col_vals.mean()

    avg_row["Row Average"] = pd.to_numeric(
        graded_df["Row Average"], errors="coerce"
    ).mean()

    all_scores = []
    for col in grade_cols:
        vals = pd.to_numeric(graded_df[col], errors="coerce").dropna()
        all_scores.extend(vals.tolist())

    overall_avg = sum(all_scores) / len(all_scores) if all_scores else None

    overall_row = {
        "Paragraph": "OVERALL_AVERAGE_ALL_CELLS",
        "GT Ligand_Name": "",
        "Pred Ligand_Name": "",
        "Match Status": ""
    }

    for col in grade_cols:
        overall_row[col] = ""

    overall_row["Row Average"] = overall_avg

    graded_df = pd.concat(
        [graded_df, pd.DataFrame([avg_row, overall_row])],
        ignore_index=True
    )

# ---------- SUMMARY ----------
summary = {
    "Total GT paragraphs": len(gt_groups),
    "Total GT rows": len(df_gt),
    "Total Pred rows": len(df_pred),
    "Matched rows": int((graded_df["Match Status"] == "matched").sum()) if not graded_df.empty else 0,
    "Missing GT ligands": int((graded_df["Match Status"] == "missing_in_prediction").sum()) if not graded_df.empty else 0,
    "Extra predicted ligands": len(extra_df),
}

numeric_row_avg = pd.to_numeric(graded_df["Row Average"], errors="coerce")
summary["Overall average score"] = numeric_row_avg.mean()

summary_df = pd.DataFrame([summary])

# ---------- FINAL SAVE ----------
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    graded_df.to_excel(writer, sheet_name="graded", index=False)
    extra_df.to_excel(writer, sheet_name="extra_pred", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)

print(f"\n✅ Finished. Saved graded Excel to: {OUTPUT_FILE}")

GT rows: 32
PRED rows: 32
GT paragraphs: 26
PRED paragraphs: 26

[Paragraph 1]
GT ligands: ['octadecyl acyclopa (ODA)']
Pred ligands: ['octadecyl acyclopa (ODA)']
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Chemical_Properties -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Radiolysis_Resistance -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Thermodynamics -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Kinetics -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Loading_Capacity -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Operational_Condition_Range -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Physical_Properties -> 0.9
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Phase_Disengagement -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Extraction_Efficiency -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Separation_Factor -> 1.0
[Paragraph 1] Matched | octadecyl acyclopa (ODA) | Dependence -> 0